# IndustrialBrain — Predictive Maintenance Model Training
**Trains on real datasets: AI4I 2020 + SKAB (Skoltech Anomaly Benchmark)**

## Datasets
- AI4I 2020 Predictive Maintenance Classification Dataset (UCI / Kaggle) — Random Forest
- SKAB: Skoltech Anomaly Benchmark (Kaggle) — Isolation Forest

## What this notebook trains
1. `failure_rf_model.pkl` — Random Forest failure classifier
2. `scaler.pkl` — StandardScaler for RF features
3. `iso_forest.pkl` — Isolation Forest anomaly detector
4. `iso_scaler.pkl` — StandardScaler for sensor features

## After training: Download models and place at:
```
ml/predictive_maintenance/models/failure_rf_model.pkl
ml/predictive_maintenance/models/scaler.pkl
ml/predictive_maintenance/models/iso_forest.pkl
ml/predictive_maintenance/models/iso_scaler.pkl
```

In [ ]:
# Install dependencies
!pip install -q kaggle scikit-learn pandas numpy joblib

In [ ]:
# Upload your kaggle.json credential
from google.colab import files
import os

print('Upload your kaggle.json file from Kaggle → Account → API → Create New Token')
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(uploaded['kaggle.json'])
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('kaggle.json installed!')

In [ ]:
# Download AI4I 2020 dataset
!mkdir -p /data/ai4i
!kaggle datasets download -d stephanmatzka/predictive-maintenance-dataset-ai4i-2020 -p /data/ai4i --unzip
!ls -la /data/ai4i

In [ ]:
# Download SKAB dataset
!mkdir -p /data/skab
!kaggle datasets download -d yuriykatser/skoltech-anomaly-benchmark-skab -p /data/skab --unzip
!ls -la /data/skab

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

MODEL_DIR = Path('/models')
MODEL_DIR.mkdir(exist_ok=True)

# Load AI4I
ai4i_files = list(Path('/data/ai4i').glob('*.csv'))
print(f'AI4I files: {ai4i_files}')
df_ai4i = pd.read_csv(ai4i_files[0])
print(f'AI4I shape: {df_ai4i.shape}')
print(df_ai4i.head(2))

In [ ]:
# ---- Train Random Forest ----

RENAME = {
    'Air temperature [K]': 'air_temperature_k',
    'Process temperature [K]': 'process_temperature_k',
    'Rotational speed [rpm]': 'rotational_speed_rpm',
    'Torque [Nm]': 'torque_nm',
    'Tool wear [min]': 'tool_wear_min',
}
df = df_ai4i.rename(columns=RENAME).copy()

rng = np.random.RandomState(42)
n = len(df)
df['days_since_maintenance'] = rng.randint(1, 365, n)
df['overdue_days'] = (df['days_since_maintenance'] > 180).astype(int) * rng.randint(0, 90, n)
df['emergency_count_6m'] = rng.randint(0, 5, n)
df['corrective_ratio'] = rng.uniform(0, 1, n)

FEAT = [
    'air_temperature_k', 'process_temperature_k', 'rotational_speed_rpm',
    'torque_nm', 'tool_wear_min', 'days_since_maintenance',
    'overdue_days', 'emergency_count_6m', 'corrective_ratio'
]

TARGET = 'Machine failure'
print(f'Class distribution:\n{df[TARGET].value_counts()}')

X = df[FEAT].fillna(df[FEAT].median())
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

rf = RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=5, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train_s, y_train)

print('\n--- Random Forest Classification Report ---')
print(classification_report(y_test, rf.predict(X_test_s)))
print('Confusion Matrix:')
print(confusion_matrix(y_test, rf.predict(X_test_s)))

joblib.dump(rf, MODEL_DIR / 'failure_rf_model.pkl')
joblib.dump(scaler, MODEL_DIR / 'scaler.pkl')
print('\n✅ Random Forest + Scaler saved!')

In [ ]:
# ---- Train Isolation Forest (SKAB) ----

skab_files = list(Path('/data/skab').glob('**/*.csv'))
print(f'Found {len(skab_files)} SKAB files')

dfs = []
for f in skab_files:
    try:
        d = pd.read_csv(f, sep=';')
        dfs.append(d)
    except:
        pass

df_skab = pd.concat(dfs, ignore_index=True)
print(f'SKAB combined shape: {df_skab.shape}')
print(df_skab.columns.tolist())
print(df_skab.head(2))

In [ ]:
SENSOR_COLS = [
    'Accelerometer1RMS', 'Accelerometer2RMS', 'Current', 'Pressure',
    'Temperature', 'Thermocouple', 'Voltage', 'Volume Flow RateRMS'
]
available = [c for c in SENSOR_COLS if c in df_skab.columns]
print(f'Available sensor columns: {available}')

X_iso = df_skab[available].fillna(0).replace([np.inf, -np.inf], 0)

iso_scaler = StandardScaler()
X_iso_s = iso_scaler.fit_transform(X_iso)

iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=42, n_jobs=-1)
iso.fit(X_iso_s)

# Evaluate if anomaly column present
if 'anomaly' in df_skab.columns:
    from sklearn.metrics import f1_score, classification_report as cr
    y_true = df_skab['anomaly'].fillna(0).astype(int)
    y_pred = (iso.predict(X_iso_s) == -1).astype(int)
    print(f'F1 Score: {f1_score(y_true, y_pred):.4f}')
    print(cr(y_true, y_pred))

joblib.dump(iso, MODEL_DIR / 'iso_forest.pkl')
joblib.dump(iso_scaler, MODEL_DIR / 'iso_scaler.pkl')
print('✅ Isolation Forest + Scaler saved!')

In [ ]:
# Zip and download models
!zip -j /models/predictive_maintenance_models.zip /models/*.pkl

from google.colab import files
files.download('/models/predictive_maintenance_models.zip')
print('\n✅ Models downloaded! Extract and place in:')
print('   ml/predictive_maintenance/models/')